# QLoRA Fine-Tuning for Multimodal Alignment of Voxtral with GLaDOS Persona

In [1]:
import torch
import pandas as pd
from datasets import Dataset, Audio
from transformers import (
    AutoProcessor,
    VoxtralForConditionalGeneration,
    BitsAndBytesConfig,
    TrainingArguments,
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
import os

device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "mistralai/Voxtral-Mini-3B-2507"
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

# 8-Bit Quantization Configuration for 12GB VRAM constraints
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=False,
)

In [2]:
print(f"Loading {model_id} in 8-bit precision...")
processor = AutoProcessor.from_pretrained(model_id)
model = VoxtralForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    torch_dtype=compute_dtype,
    device_map="auto"
)

# Prepare model for gradient training
model = prepare_model_for_kbit_training(model)

Loading mistralai/Voxtral-Mini-3B-2507 in 8-bit precision...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

#### Dataset Formatting for Multimodal SFT

In [3]:
def format_voxtral_dataset(csv_path, audio_dir):
    df = pd.read_csv(csv_path)
    # Drop rows missing critical components
    df = df.dropna(subset=['Assistant_Payload', 'Target_GLaDOS_Response', 'audio_file_name'])
    dataset_dict = {"messages": []}
    for _, row in df.iterrows():
        full_audio_path = os.path.join(audio_dir, row["audio_file_name"])
        # Validation check to prevent the trainer from crashing mid-epoch
        if not os.path.exists(full_audio_path):
            print(f"Warning: Skipping {row['audio_file_name']} - File not found at {full_audio_path}")
            continue
        target_output = f"{row['Assistant_Payload']}\n\n{row['Target_GLaDOS_Response']}{processor.tokenizer.eos_token}"
        conversation = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "audio",
                        "path": full_audio_path # The processor will handle loading and feature extraction during training
                    }
                ]
            },
            {
                "role": "assistant",
                "content": [
                    {
                        "type": "text",
                        "text": target_output # Wrapped in list to satisfy Arrow schema
                    }
                ]
            }
        ]
        dataset_dict["messages"].append(conversation)
    # No need to cast to datasets.Audio(), saving massive amounts of RAM
    return Dataset.from_dict(dataset_dict)

print("Formatting multimodal dataset...")
train_dataset = format_voxtral_dataset("./data/combined_multimodal_dataset_test.csv", "./data/synthesized_test/")

Formatting multimodal dataset...


#### Supervised Fine-Tuning with TRL's SFTTrainer

In [4]:
def voxtral_collate_fn(batch):
    conversations = [item["messages"] for item in batch]
    # Let the processor handle the audio loading, padding, and tokenization.
    # But we need to ensure the underlying Mistral tokenizer knows this is for training.
    inputs = processor.apply_chat_template(
        conversations,
        tokenize=True,
        return_dict=True,
        continue_final_message=True,
        processor_kwargs={
            "padding": True,
            "return_tensors": "pt"
        },
    )
    labels = inputs["input_ids"].clone()
    # 1. Mask standard padding tokens
    labels[labels == processor.tokenizer.pad_token_id] = -100
    # 2. Mask User Input (The Audio & Prompt) so loss is ONLY calculated on the Assistant's JSON/Text
    # Mistral/Voxtral uses the [/INST] tag to separate user prompt from assistant response
    inst_token_id = processor.tokenizer.convert_tokens_to_ids("[/INST]")
    for i in range(labels.shape[0]):
        try:
            # Find the last occurrence of [/INST] (end of user instruction)
            inst_indices = (labels[i] == inst_token_id).nonzero(as_tuple=True)[0]
            if len(inst_indices) > 0:
                inst_idx = inst_indices[-1]
                # Mask everything up to and including [/INST]
                # This ensures the model is ONLY graded on the Assistant's text
                labels[i, :inst_idx + 1] = -100
        except IndexError:
            pass # Fallback in case of formatting anomaly
    inputs["labels"] = labels
    return inputs

model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.eos_token_id
model.config.bos_token_id = processor.tokenizer.bos_token_id

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

training_args = SFTConfig(
    output_dir="./models/voxtral-glados-sft",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=500, # Adjust based on dataset size
    save_strategy="epoch",
    optim="adamw_bnb_8bit", # paged_adamw_8bit or adamw_bnb_8bit
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    remove_unused_columns=False, # Crucial so the collator receives the dicts
    dataset_kwargs={"skip_prepare_dataset": True}
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=voxtral_collate_fn,
    peft_config=lora_config,
)
trainer.model.print_trainable_parameters()

trainable params: 30,965,760 || all params: 4,707,236,864 || trainable%: 0.6578


In [5]:
print("Initiating QLoRA Multimodal Alignment...")
trainer.train()

# Save the final adapter weights
trainer.model.save_pretrained("./models/voxtral-glados-final-adapters")
processor.save_pretrained("./models/voxtral-glados-final-adapters")
print("Training complete. Adapters saved.")

Initiating QLoRA Multimodal Alignment...


InvalidMessageStructureException: Expected last role User or Tool (or Assistant with prefix or continue_final_message set to True) for serving but got assistant